# Pueblo, Colorado — LVT Policy Analysis Template

This notebook sets up a city-level Land Value Tax (LVT) modeling workflow for **Pueblo, Colorado**.

> Status: **Template ready for data-source confirmation.**
>
> Before running this end-to-end, confirm the ArcGIS parcel layer URL, layer ID, levy scope (city-only vs full stack), and exemption fields.


## Stage A/B policy assumptions (to confirm)

Following `LVT_MODELING_GUIDE.md`, this notebook defaults to preserving Pueblo's current property tax structure except for the land/building rate shift.

- Policy scope: **City levy** (default assumption; update if needed)
- Reform type: **Split-rate LVT**
- Keep existing exemptions/abatements: **Yes**
- Keep post-tax credits/rollbacks: **Yes** (if present in source)
- Intentional departures from current law: **None unless explicitly added below**


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from cloud_utils import get_feature_data_with_geometry
from lvt_utils import (
    calculate_current_tax,
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
)
from viz import create_spokane_property_category_chart, create_threshold_change_chart


In [ ]:
# Configuration
REPO_ROOT = Path(os.getcwd()).resolve().parent if Path(os.getcwd()).name == "examples" else Path(os.getcwd()).resolve()
data_dir = REPO_ROOT / "examples" / "data" / "pueblo"
data_dir.mkdir(parents=True, exist_ok=True)

# TODO: Confirm Pueblo parcel ArcGIS service details.
# Suggested workflow: inspect Pueblo County / City of Pueblo open data portal and update these values.
BASE_URL = "https://<pueblo-arcgis-host>/arcgis/rest/services/<parcel-dataset>/FeatureServer"
LAYER_ID = 0
WHERE_CLAUSE = "1=1"

# Baseline policy assumption (change if policy discussion specifies otherwise)
LAND_IMPROVEMENT_RATIO = 2.0


## Step 1: Load parcel data

If cached data exists, load it. Otherwise pull from ArcGIS using `get_feature_data_with_geometry()`.


In [ ]:
cache_path = data_dir / "pueblo_parcels_raw.parquet"

if cache_path.exists():
    gdf = gpd.read_parquet(cache_path)
    print(f"Loaded cached parcels: {len(gdf):,}")
else:
    if "<pueblo-arcgis-host>" in BASE_URL:
        raise ValueError("Update BASE_URL/LAYER_ID before scraping Pueblo parcel data.")

    gdf = get_feature_data_with_geometry(
        dataset_name="pueblo_parcels",
        base_url=BASE_URL,
        layer_id=LAYER_ID,
        where=WHERE_CLAUSE,
        paginate=True,
    )
    gdf.to_parquet(cache_path, index=False)
    print(f"Fetched and cached parcels: {len(gdf):,}")

print(gdf.shape)
print(gdf.columns.tolist()[:40])


## Step 2: Column mapping (update once source is confirmed)

Replace the placeholder field names with Pueblo-specific names for:
- Parcel ID
- Land value
- Improvement value
- Total/taxable value
- Current millage/rate (or derived city levy)
- Exemptions / exemption flags
- Property use classification


In [ ]:
# TODO: Replace these placeholders with real Pueblo column names.
COL_PARCEL_ID = "PARCEL_ID"
COL_CITY_NAME = "SITUS_CITY"
COL_LAND_VALUE = "LAND_VALUE"
COL_IMP_VALUE = "IMPROVEMENT_VALUE"
COL_TOTAL_VALUE = "TOTAL_VALUE"
COL_MILLAGE = "CITY_MILLAGE"
COL_EXEMPT_AMT = None
COL_EXEMPT_FLAG = None
COL_USE = "PROPERTY_USE"

required_cols = [COL_PARCEL_ID, COL_LAND_VALUE, COL_IMP_VALUE, COL_TOTAL_VALUE, COL_MILLAGE, COL_USE]
missing = [c for c in required_cols if c is not None and c not in gdf.columns]
if missing:
    print("Missing required columns (expected before mapping is finalized):")
    print(missing)


## Step 3: Filter to Pueblo city parcels

If source is county-wide, filter to parcels inside city limits.


In [ ]:
df = gdf.copy()

if COL_CITY_NAME in df.columns:
    city_mask = df[COL_CITY_NAME].astype(str).str.upper().str.contains("PUEBLO", na=False)
    df = df[city_mask].copy()

print(f"Rows after city filter: {len(df):,}")


## Step 4: Normalize numeric columns and exemptions


In [ ]:
for col in [COL_LAND_VALUE, COL_IMP_VALUE, COL_TOTAL_VALUE, COL_MILLAGE]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

if COL_EXEMPT_AMT is not None and COL_EXEMPT_AMT in df.columns:
    df[COL_EXEMPT_AMT] = pd.to_numeric(df[COL_EXEMPT_AMT], errors="coerce").fillna(0)

if COL_EXEMPT_FLAG is None:
    df["full_exmp"] = 0
    exemption_flag_col = "full_exmp"
else:
    exemption_flag_col = COL_EXEMPT_FLAG


## Step 5: Property category mapping

Update `category_map` to match Pueblo's assessor use codes/descriptions.


In [ ]:
category_map = {
    "SINGLE FAMILY": "Single Family Residential",
    "MULTI-FAMILY": "Large Multi-Family (5+ units)",
    "DUPLEX": "Small Multi-Family (2-4 units)",
    "TRIPLEX": "Small Multi-Family (2-4 units)",
    "QUADPLEX": "Small Multi-Family (2-4 units)",
    "VACANT": "Vacant Land",
    "COMMERCIAL": "Commercial",
    "INDUSTRIAL": "Industrial",
}

if COL_USE in df.columns:
    use_series = df[COL_USE].astype(str).str.upper()
    df["PROPERTY_CATEGORY"] = "Other"
    for key, value in category_map.items():
        df.loc[use_series.str.contains(key, na=False), "PROPERTY_CATEGORY"] = value
else:
    df["PROPERTY_CATEGORY"] = "Other"

print(df["PROPERTY_CATEGORY"].value_counts().head(15))


## Step 6: Calculate current tax baseline


In [ ]:
if COL_TOTAL_VALUE not in df.columns or COL_MILLAGE not in df.columns:
    raise ValueError("Update column mapping before calculating current tax.")

current_revenue, _, df = calculate_current_tax(
    df=df,
    tax_value_col=COL_TOTAL_VALUE,
    millage_rate_col=COL_MILLAGE,
    exemption_col=COL_EXEMPT_AMT,
    exemption_flag_col=exemption_flag_col,
    land_value_col=COL_LAND_VALUE if COL_LAND_VALUE in df.columns else None,
    improvement_value_col=COL_IMP_VALUE if COL_IMP_VALUE in df.columns else None,
)

print(f"Current modeled revenue: ${current_revenue:,.0f}")
print(df[["current_tax", "current_taxable_value"]].describe())


## Step 7: Run split-rate LVT scenario (default 2:1)


In [ ]:
land_millage, imp_millage, new_revenue, df = model_split_rate_tax(
    df=df,
    land_value_col=COL_LAND_VALUE,
    improvement_value_col=COL_IMP_VALUE,
    current_revenue=current_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    exemption_col=COL_EXEMPT_AMT,
    exemption_flag_col=exemption_flag_col,
)

print(f"Land millage: {land_millage:.4f}")
print(f"Improvement millage: {imp_millage:.4f}")
print(f"New modeled revenue: ${new_revenue:,.0f}")


## Step 8: Summaries and charts


In [ ]:
category_summary = calculate_category_tax_summary(
    df=df,
    category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
    pct_threshold=10.0,
)

print_category_tax_summary(category_summary, title="Pueblo 2:1 LVT — Category Summary")

create_spokane_property_category_chart(
    summary_df=category_summary,
    title="Pueblo, CO 2:1 Split-Rate Tax Impact by Property Category",
    min_count=10,
)

create_threshold_change_chart(
    summary_df=category_summary,
    title="Pueblo, CO Share of Parcels with Tax Changes Greater than 10%",
    threshold=10.0,
    min_count=10,
)


## Step 9: Save modeled output


In [ ]:
out_path = data_dir / "pueblo_modeled_2to1.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved modeled output: {out_path}")


## Notes for completion

- Confirm Pueblo data source URL + layer metadata.
- Finalize field mapping and exemption logic based on Pueblo assessor schema.
- Validate current modeled revenue against an official Pueblo levy benchmark.
- Optionally add Census equity analysis once parcel geography is validated.
